In [6]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.utils.features import split_features

In [7]:
DATA_DIR = Path("../datasets/final/ml")

DATASET_NAMES = [
    "cicids2017",
    "unsw_nb15",
    "iot23",
]

ABLATION_FEATURES = [
    "bidirectional_min_ps",
    "bidirectional_min_piat_ms",
]

RANDOM_STATE = 1
N_SPLITS = 5

In [8]:
def build_logistic_pipeline(numerical_features, categorical_features):
    preprocessor = ColumnTransformer(
        transformers=[
            ("numerical", StandardScaler(), numerical_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(
                solver="lbfgs",
                max_iter=1000,
            )),
        ]
    )

In [9]:
def summarize_coefficients(estimators):
    coefficients_by_fold = []

    for fold, estimator in enumerate(estimators, start=1):
        preprocessor = estimator.named_steps["preprocessor"]
        model = estimator.named_steps["model"]

        feature_names = [
            name.removeprefix("numerical__").removeprefix("categorical__")
            for name in preprocessor.get_feature_names_out()
        ]

        coefficients_by_fold.append(pd.DataFrame({
            "fold": fold,
            "feature": feature_names,
            "coefficient": model.coef_[0],
        }))

    coefficients = pd.concat(coefficients_by_fold, ignore_index=True)
    summary = (
        coefficients
        .groupby("feature", as_index=False)
        .agg(
            coefficient_mean=("coefficient", "mean"),
            coefficient_std=("coefficient", "std"),
            mean_abs_coefficient=("coefficient", lambda values: values.abs().mean()),
        )
    )

    return summary.sort_values("mean_abs_coefficient", ascending=False)


def evaluate_configuration(
    df,
    target,
    numerical_features,
    categorical_features,
    cross_validation_splits,
    return_estimator=False,
):
    feature_columns = numerical_features + categorical_features

    return cross_validate(
        build_logistic_pipeline(
            numerical_features,
            categorical_features,
        ),
        df[feature_columns],
        target,
        cv=cross_validation_splits,
        scoring="f1_macro",
        n_jobs=1,
        return_estimator=return_estimator,
    )


def evaluate_training_dataset(dataset_name):
    df = pd.read_parquet(
        DATA_DIR / f"{dataset_name}_train.parquet"
    )

    numerical_features, categorical_features = split_features(df)
    target = df["label_binary"].eq("ATTACK").astype(int)

    cross_validation = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    cross_validation_splits = list(cross_validation.split(df, target))

    baseline_scores = evaluate_configuration(
        df,
        target,
        numerical_features,
        categorical_features,
        cross_validation_splits,
        return_estimator=True,
    )
    baseline_f1_macro = baseline_scores["test_score"]

    results = [{
        "dataset": dataset_name,
        "configuration": "selected_features",
        "added_feature": None,
        "numerical_features": len(numerical_features),
        "categorical_features": len(categorical_features),
        "f1_macro_mean": baseline_f1_macro.mean(),
        "f1_macro_std": baseline_f1_macro.std(),
        "f1_macro_delta_mean": 0.0,
        "f1_macro_delta_std": 0.0,
    }]

    for feature in ABLATION_FEATURES:
        if feature not in df.columns:
            raise KeyError(f"Atributo ausente no dataset: {feature}")

        ablation_numerical_features = numerical_features + [feature]
        ablation_scores = evaluate_configuration(
            df,
            target,
            ablation_numerical_features,
            categorical_features,
            cross_validation_splits,
        )
        ablation_f1_macro = ablation_scores["test_score"]
        paired_delta = ablation_f1_macro - baseline_f1_macro

        results.append({
            "dataset": dataset_name,
            "configuration": f"with_{feature}",
            "added_feature": feature,
            "numerical_features": len(ablation_numerical_features),
            "categorical_features": len(categorical_features),
            "f1_macro_mean": ablation_f1_macro.mean(),
            "f1_macro_std": ablation_f1_macro.std(),
            "f1_macro_delta_mean": paired_delta.mean(),
            "f1_macro_delta_std": paired_delta.std(),
        })

    coefficient_summary = summarize_coefficients(
        baseline_scores["estimator"]
    )

    return results, coefficient_summary

In [10]:
ablation_results = []
coefficient_summaries = {}

for dataset_name in DATASET_NAMES:
    dataset_results, coefficient_summary = evaluate_training_dataset(dataset_name)
    ablation_results.extend(dataset_results)
    coefficient_summaries[dataset_name] = coefficient_summary

ablation_results_df = pd.DataFrame(ablation_results)

display(ablation_results_df)

for dataset_name in DATASET_NAMES:
    print(f"\nCoeficientes - {dataset_name}")
    display(coefficient_summaries[dataset_name])

,dataset,configuration,added_feature,numerical_features,categorical_features,f1_macro_mean,f1_macro_std,f1_macro_delta_mean,f1_macro_delta_std
0,cicids2017,selected_features,NaN,14,1,0.933720,0.001031,0.000000,0.000000
1,cicids2017,with_bidirectional_min_ps,bidirectional_min_ps,15,1,0.933766,0.000916,0.000046,0.000124
2,cicids2017,with_bidirectional_min_piat_ms,bidirectional_min_piat_ms,15,1,0.934146,0.000840,0.000426,0.000240
3,unsw_nb15,selected_features,NaN,14,1,0.920183,0.001278,0.000000,0.000000
4,unsw_nb15,with_bidirectional_min_ps,bidirectional_min_ps,15,1,0.919877,0.001396,-0.000306,0.001030
5,unsw_nb15,with_bidirectional_min_piat_ms,bidirectional_min_piat_ms,15,1,0.920498,0.000769,0.000315,0.000537
6,iot23,selected_features,NaN,14,1,0.995927,0.000235,0.000000,0.000000
7,iot23,with_bidirectional_min_ps,bidirectional_min_ps,15,1,0.995980,0.000205,0.000053,0.000077
8,iot23,with_bidirectional_min_piat_ms,bidirectional_min_piat_ms,15,1,0.995821,0.000364,-0.000106,0.000322



Coeficientes - cicids2017


,feature,coefficient_mean,coefficient_std,mean_abs_coefficient
14,protocol_17,-5.403285,0.276139,5.403285
12,bidirectional_stddev_ps,4.572518,0.102416,4.572518
9,bidirectional_psh_packets,-4.323078,1.006442,4.323078
5,bidirectional_max_ps,-1.914437,0.084277,1.914437
15,protocol_6,1.781002,0.122083,1.781002
10,bidirectional_rst_packets,1.404435,0.005954,1.404435
3,bidirectional_fin_packets,-1.335744,0.022588,1.335744
7,bidirectional_mean_ps,-1.178940,0.031025,1.178940
4,bidirectional_max_piat_ms,-0.785020,0.016112,0.785020
13,bidirectional_syn_packets,0.719555,0.017478,0.719555



Coeficientes - unsw_nb15


,feature,coefficient_mean,coefficient_std,mean_abs_coefficient
0,bidirectional_ack_packets,51.925409,1.327665,51.925409
9,bidirectional_psh_packets,-44.548629,0.582056,44.548629
1,bidirectional_bytes,-9.331505,1.252018,9.331505
8,bidirectional_packets,6.819588,1.206438,6.819588
12,bidirectional_stddev_ps,-4.674984,0.093284,4.674984
5,bidirectional_max_ps,3.468287,0.085295,3.468287
16,protocol_89,-2.985556,0.144111,2.985556
6,bidirectional_mean_piat_ms,0.966197,0.402046,0.966197
11,bidirectional_stddev_piat_ms,-0.905626,0.429156,0.905626
3,bidirectional_fin_packets,0.784662,0.066501,0.784662



Coeficientes - iot23


,feature,coefficient_mean,coefficient_std,mean_abs_coefficient
6,bidirectional_mean_piat_ms,-15.841133,0.250719,15.841133
14,protocol_17,-8.947129,0.659216,8.947129
2,bidirectional_duration_ms,5.979468,0.226047,5.979468
9,bidirectional_psh_packets,5.905994,0.576817,5.905994
4,bidirectional_max_piat_ms,4.382291,0.589770,4.382291
15,protocol_6,4.040690,0.282687,4.040690
3,bidirectional_fin_packets,3.819947,0.775011,3.819947
0,bidirectional_ack_packets,-3.286300,0.358996,3.286300
8,bidirectional_packets,-2.545866,0.216069,2.545866
5,bidirectional_max_ps,2.281348,0.455180,2.281348
